# Notebook 02 — Preprocessing and Feature Engineering

Second stage of the MoEForecaster pipeline (Mixture-of-Experts forecasting of the hourly electricity demand of Colombia's National Interconnected System, SIN).

---

## Objective

Transform the raw hourly demand series downloaded in notebook 01 into a feature dataset for the clustering, classification and expert models of the following notebooks. The notebook:

1. Loads the raw series and places it on a regular hourly index.
2. Builds **38 features** in five groups: cyclic calendar encodings (6), Colombian holiday features (4), Fourier terms for the daily and weekly cycles (10), autoregressive lags (12) and rolling statistics (6).
3. Assesses the stationarity of the demand series (ADF and KPSS tests) and of the residual of a multi-seasonal MSTL decomposition (periods 24 and 168 h).
4. Saves the feature matrix and the list of feature names per group.

All features are **causal**: calendar, holiday and Fourier features are deterministic functions of the timestamp, and the autoregressive features use only past demand values, so there is no data leakage from the target into the features. No outlier treatment or imputation is applied in this notebook; the raw values are used as downloaded.

**Place in the pipeline:** 01 data acquisition → 02 preprocessing and feature engineering (this notebook) → 03 clustering of daily and weekly profiles → 04 cluster analysis and cluster classifiers → 05 expert selection → 06 gating design → 07 hyperparameter optimization → 08 production assembly → 09 benchmark.

## Inputs

| File | Content | Produced by |
|---|---|---|
| `data/raw/demand.parquet` | Hourly SIN demand, columns `datetime`, `demand_kwh` | Notebook 01 |

## Outputs

| File | Content | Used by |
|---|---|---|
| `data/processed/features.parquet` | Target `demand` + 38 features, hourly `datetime` index (52,608 × 39 for the full 2020–2025 period). Initial rows with NaN from the lag/rolling warm-up are kept. | Notebooks 03 onwards |
| `data/processed/feature_names.json` | Dictionary mapping each feature group (`target`, `calendar`, `holiday`, `fourier`, `autoregressive_lags`, `autoregressive_rolling`) to its column names | Documentation of the feature set (not read by the later notebooks, which load `features.parquet` directly) |
| `results/02_mstl_decomposition.png`, `results/02_mstl_decomposition_jan2022.png`, `results/02_mstl_residual_diagnostics.png` | MSTL decomposition and residual diagnostics (Section 5) | Documentation |

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

---
## 1. Configuration

In [ ]:
import os
# ============================================================
# CONFIGURATION
# ============================================================
RANDOM_SEED  = 42
N_JOBS       = -1

---
## 2. Imports and plotting style

In [ ]:
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import interpolate as sp_interp
from scipy import stats as sp_stats
from statsmodels.tsa.seasonal import STL, MSTL
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Default Matplotlib configuration ─────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (14, 4),
    "figure.dpi": 120,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "font.family": "serif",
    "axes.grid": True,
    "grid.alpha": 0.3,
})
sns.set_palette("colorblind")

---
## 3. Data loading

The raw series written by notebook 01 is loaded, the timestamp column is set as a sorted `DatetimeIndex`, the demand column (`demand_kwh`) is renamed to `demand`, and the series is put on a regular hourly frequency with `asfreq("h")`. Any hour missing from the raw file would appear as NaN after this step; the number of missing values is reported.

In [ ]:
# ── Load the raw demand series ───────────────────────────────────────────────
raw_path = Path(DATA_DIR, "raw", "demand.parquet")
df = pd.read_parquet(raw_path)
print(f"Original shape: {df.shape}")
print(f"Date range: {df.index.min()} → {df.index.max()}" if isinstance(df.index, pd.DatetimeIndex) else "")
df.head()

In [ ]:
# ── Ensure a datetime index ──────────────────────────────────────────────────
# Adapt column names if needed: a date/time column and a demand column are expected.
if not isinstance(df.index, pd.DatetimeIndex):
    # Try to find a datetime column ("fecha" is the Spanish word for date)
    date_cols = [c for c in df.columns if "date" in c.lower() or "fecha" in c.lower() or "time" in c.lower()]
    if date_cols:
        df[date_cols[0]] = pd.to_datetime(df[date_cols[0]])
        df = df.set_index(date_cols[0]).sort_index()
    else:
        raise ValueError("No datetime column found. Adjust the loading logic.")

# Identify the demand column (case-insensitive search; "demanda" is the Spanish word for demand)
demand_col_candidates = [c for c in df.columns if "demand" in c.lower() or "demanda" in c.lower()]
if len(demand_col_candidates) == 0:
    # Fallback: use the first numeric column
    demand_col_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
DEMAND_COL = demand_col_candidates[0]
print(f"Using demand column: '{DEMAND_COL}'")

# Rename for consistency
df = df.rename(columns={DEMAND_COL: "demand"})
DEMAND_COL = "demand"

# Keep only the demand column for this preprocessing stage
demand_series = df["demand"].copy()
demand_series.index.name = "datetime"
demand_series = demand_series.asfreq("h")  # enforce hourly frequency

print(f"\nSeries length  : {len(demand_series):,} hours")
print(f"Missing values : {demand_series.isna().sum():,} ({demand_series.isna().mean()*100:.2f} %)")
print(f"Date range     : {demand_series.index.min()} → {demand_series.index.max()}")

---
## 4. Feature engineering (production features only)

Every feature computed here **must be available at inference time**, so that the same features can be built when the system is used in production. This means:
- Calendar, holiday and Fourier features are deterministic: they are known for any future timestamp.
- Autoregressive features use only **past** demand values (strict causality).
- Rolling statistics are computed on demand shifted by one hour and with `center=False` (backward-looking window), so the window ends at $t-1$.

The feature names are collected per group in the dictionary `feature_names`, which is saved to `feature_names.json` (Section 6).

In [ ]:
# ── 4.0  Build the feature DataFrame ─────────────────────────────────────────

feat = pd.DataFrame({"demand": demand_series}, index=demand_series.index)
feat.index.name = "datetime"

# Feature names are collected by group for the JSON export.
feature_names: dict[str, list[str]] = {
    "target": ["demand"],
    "calendar": [],
    "holiday": [],
    "fourier": [],
    "autoregressive_lags": [],
    "autoregressive_rolling": [],
}

### 4.1 Calendar features (cyclic encoding)

Hour of the day (period 24), day of the week (period 7) and month (period 12) are encoded with sine/cosine pairs, $\sin(2\pi x/P)$ and $\cos(2\pi x/P)$. The cyclic encoding preserves the circular continuity of these variables (e.g. hour 23 is close to hour 0, December is close to January), which an integer encoding would break.

In [ ]:
# ── 4.1  Calendar features — cyclic sine/cosine encoding ─────────────────────

feat["hour_sin"]      = np.sin(2 * np.pi * feat.index.hour / 24)
feat["hour_cos"]      = np.cos(2 * np.pi * feat.index.hour / 24)
feat["dayofweek_sin"] = np.sin(2 * np.pi * feat.index.dayofweek / 7)
feat["dayofweek_cos"] = np.cos(2 * np.pi * feat.index.dayofweek / 7)
feat["month_sin"]     = np.sin(2 * np.pi * feat.index.month / 12)
feat["month_cos"]     = np.cos(2 * np.pi * feat.index.month / 12)

calendar_cols = ["hour_sin", "hour_cos", "dayofweek_sin", "dayofweek_cos", "month_sin", "month_cos"]
feature_names["calendar"].extend(calendar_cols)

print(f"Calendar features added: {calendar_cols}")

### 4.2 Colombian holidays (Law 51 of 1983)

Colombian *Law 51 of 1983* ("Ley Emiliani") moves several holidays to the following Monday, creating long weekends. The `holidays.Colombia()` calendar of the `holidays` library implements these rules; it is built for the years of the series plus a one-year margin on each side.

Four holiday features are built:
- `is_holiday`: 1 if the date is a Colombian public holiday.
- `is_bridge`: 1 for **bridge days**, which typically show reduced demand. In the code, a bridge day is the Friday or the Saturday before a Monday holiday, or the Saturday after a Friday holiday.
- `days_to_next_holiday` / `days_from_prev_holiday`: number of days to the next holiday and since the previous holiday (0 on the holiday itself), clipped at 30 days to avoid very large values.

In [ ]:
# ── 4.2  Holiday features ────────────────────────────────────────────────────

# Holiday calendar covering the full date range + a 1-year margin
year_min = feat.index.year.min() - 1
year_max = feat.index.year.max() + 1
co_holidays = holidays.Colombia(years=range(year_min, year_max + 1))
holiday_dates = set(co_holidays.keys())

# is_holiday
# feat.index.date is converted to a pandas Series before applying .isin()
feat["is_holiday"] = pd.Series(feat.index.date, index=feat.index).isin(
    holiday_dates
).astype(int)

# is_bridge: Friday/Saturday before a Monday holiday, or Saturday after a Friday holiday
dates_series = pd.Series(feat.index.date).drop_duplicates().reset_index(drop=True)
bridge_dates = set()
for hd in holiday_dates:
    hd_pd = pd.Timestamp(hd)
    # If the holiday falls on a Monday, the preceding Friday and Saturday are bridge days
    if hd_pd.weekday() == 0:  # Monday
        bridge_dates.add((hd_pd - pd.Timedelta(days=3)).date())  # Friday
        bridge_dates.add((hd_pd - pd.Timedelta(days=2)).date())  # Saturday
    # If the holiday falls on a Friday, the following Saturday is a bridge day
    elif hd_pd.weekday() == 4:  # Friday
        bridge_dates.add((hd_pd + pd.Timedelta(days=1)).date())  # Saturday

feat["is_bridge"] = pd.Series(
    feat.index.date, index=feat.index
).isin(bridge_dates).astype(int)

# days_to_next_holiday / days_from_prev_holiday
sorted_holidays = sorted(holiday_dates)
holiday_ts = pd.to_datetime(sorted_holidays)


def days_to_nearest(dt, holiday_list, direction="next") -> int:
    """Return the number of days to the next holiday or since the previous holiday."""
    d = dt.date() if hasattr(dt, "date") else dt
    d = pd.Timestamp(d)
    if direction == "next":
        future = holiday_list[holiday_list >= d]
        return (future[0] - d).days if len(future) > 0 else 365
    else:
        past = holiday_list[holiday_list <= d]
        return (d - past[-1]).days if len(past) > 0 else 365


# Compute once per unique date and map back (much faster than per row)
unique_dates = pd.Series(feat.index.date).unique()
days_next = {d: days_to_nearest(d, holiday_ts, "next") for d in unique_dates}
days_prev = {d: days_to_nearest(d, holiday_ts, "prev") for d in unique_dates}

feat["days_to_next_holiday"]   = pd.Series(feat.index.date, index=feat.index).map(days_next).astype(int)
feat["days_from_prev_holiday"] = pd.Series(feat.index.date, index=feat.index).map(days_prev).astype(int)

# Clip to a reasonable maximum to avoid very large values
feat["days_to_next_holiday"]   = feat["days_to_next_holiday"].clip(upper=30)
feat["days_from_prev_holiday"] = feat["days_from_prev_holiday"].clip(upper=30)

holiday_cols = ["is_holiday", "is_bridge", "days_to_next_holiday", "days_from_prev_holiday"]
feature_names["holiday"].extend(holiday_cols)

print(f"Holiday features added: {holiday_cols}")
print(f"Total holidays in the range: {feat['is_holiday'].sum() // 24} days")
print(f"Total bridge days: {feat['is_bridge'].sum() // 24} days")

### 4.3 Fourier terms

Sine/cosine terms $\sin(2\pi k t/P)$ and $\cos(2\pi k t/P)$ capture the daily cycle ($P = 24$ h, harmonics $k = 1, 2, 3$) and the weekly cycle ($P = 168$ h, harmonics $k = 1, 2$), giving 10 features. Here $t$ is the number of hours elapsed since the first timestamp of the series (row position). The higher harmonics allow smooth, non-sinusoidal intra-day and intra-week shapes to be represented, complementing the calendar encoding of Section 4.1.

In [ ]:
# ── 4.3  Fourier terms ───────────────────────────────────────────────────────
# Capture the daily (24 h, 3 harmonics) and weekly (168 h, 2 harmonics) seasonality.
# t = hours elapsed since the start of the series.

FOURIER_TERMS = {24: 3, 168: 2}  # period (h) → number of harmonics

t = np.arange(len(feat))
for period, n_harmonics in FOURIER_TERMS.items():
    for k in range(1, n_harmonics + 1):
        for fname, func in (("sin", np.sin), ("cos", np.cos)):
            col_name = f"fourier_{period}_{fname}_{k}"
            feat[col_name] = func(2 * np.pi * k * t / period)
            feature_names["fourier"].append(col_name)

print(f"Fourier terms added ({len(feature_names['fourier'])}): {feature_names['fourier']}")

### 4.4 Autoregressive features — lags

The lag set, motivated by the ACF/PACF peaks at 24 and 168 h observed in notebook 01, covers:
- **Recent hours**: 1, 2, 3 (short-term momentum)
- **Same hour yesterday ± 1**: 23, 24, 25 (daily cycle)
- **Same hour two days ago ± 1**: 47, 48, 49
- **Same hour last week ± 1**: 167, 168, 169 (weekly cycle)

Each lag feature is `demand` shifted forward by the lag, so it only contains past values. The first 169 rows have at least one NaN lag (warm-up period).

In [ ]:
# ── 4.4  Lag features ────────────────────────────────────────────────────────

LAG_SET = [1, 2, 3, 23, 24, 25, 47, 48, 49, 167, 168, 169]

for lag in LAG_SET:
    col_name = f"demand_lag_{lag}"
    feat[col_name] = feat["demand"].shift(lag)
    feature_names["autoregressive_lags"].append(col_name)

print(f"Lag features added ({len(LAG_SET)}): lags = {LAG_SET}")

### 4.5 Rolling statistics

Rolling mean and standard deviation of demand over windows of 24 h (one day), 168 h (one week) and 336 h (two weeks), giving 6 features.

**Critical for causality:** the demand series is first shifted by one hour and the window uses `center=False` (backward-looking), so the statistic at time $t$ uses only demand up to $t-1$. Without the shift the window would include the value being forecast (target leakage). With `min_periods=1` the statistics are computed on the available history during the warm-up period (the rolling mean is NaN only in the first row and the rolling standard deviation in the first two rows).

In [ ]:
# ── 4.5  Rolling mean / standard deviation — ALWAYS center=False ─────────────

ROLLING_WINDOWS = [24, 168, 336]

for w in ROLLING_WINDOWS:
    mean_col = f"demand_rmean_{w}"
    std_col  = f"demand_rstd_{w}"
    # shift(1): the window ends at t-1. Without the shift it would include the demand
    # of the hour being forecast (leakage of the target into the features).
    past_demand = feat["demand"].shift(1)
    feat[mean_col] = past_demand.rolling(window=w, min_periods=1, center=False).mean()
    feat[std_col]  = past_demand.rolling(window=w, min_periods=1, center=False).std()
    feature_names["autoregressive_rolling"].extend([mean_col, std_col])

print(f"Rolling features added: windows = {ROLLING_WINDOWS} (mean + standard deviation for each)")

### 4.6 Feature summary

Number of features per group and number of NaN values per column (from the warm-up of the lag and rolling features, and from any missing hour in the raw data).

In [ ]:
# ── 4.6  Feature summary ─────────────────────────────────────────────────────

total_features = sum(len(v) for v in feature_names.values()) - 1  # exclude the target
print(f"\nTotal features: {total_features}")
print("\nFeatures per group:")
for cat, names in feature_names.items():
    print(f"  {cat:30s} → {len(names):3d} features")

print(f"\nDataFrame shape: {feat.shape}")
print(f"NaNs per column (top 10):")
na_counts = feat.isna().sum().sort_values(ascending=False)
print(na_counts[na_counts > 0].head(10).to_string())

---
## 5. Stationarity analysis

The stationarity of the demand series is assessed with two complementary tests (5% significance level):

| Test | H₀ | Interpretation |
|--------|----|----------------|
| **ADF** (Augmented Dickey-Fuller, lag order selected by AIC) | The series has a unit root (non-stationary) | Reject H₀ (p < 0.05) → stationary |
| **KPSS** (level stationarity, `regression="c"`) | The series is stationary | Reject H₀ (p ≤ 0.05) → non-stationary |

Using both tests together avoids the ambiguity of a single test; when they disagree, the joint interpretation (trend-stationary or difference-stationary) is printed. The tests are first applied to the demand series loaded in Section 3 (no imputation applied; NaNs, if any, are dropped). Then a **MSTL decomposition** (Multiple Seasonal-Trend decomposition using LOESS, robust fitting) with periods 24 and 168 h is computed to inspect visually whether the residual component is approximately stationary, and the same tests and a normality check are applied to that residual.

This section is diagnostic only: it does not modify the features computed in Section 4.

In [ ]:
# ── 5.1  ADF and KPSS tests ──────────────────────────────────────────────────

def stationarity_tests(series: pd.Series, name: str = "Series") -> pd.DataFrame:
    """Run the ADF and KPSS tests and return a summary DataFrame."""
    results = []

    # ADF test
    adf_stat, adf_pval, adf_lags, adf_nobs, adf_crit, _ = adfuller(series.dropna(), autolag="AIC")
    results.append({
        "Test": "ADF",
        "Statistic": f"{adf_stat:.4f}",
        "p-value": f"{adf_pval:.6f}",
        "Lags used": adf_lags,
        "H₀": "Unit root (non-stationary)",
        "Conclusion": "Stationary" if adf_pval < 0.05 else "Non-stationary",
    })

    # KPSS test
    kpss_stat, kpss_pval, kpss_lags, kpss_crit = kpss(series.dropna(), regression="c", nlags="auto")
    results.append({
        "Test": "KPSS",
        "Statistic": f"{kpss_stat:.4f}",
        "p-value": f"{kpss_pval:.6f}",
        "Lags used": kpss_lags,
        "H₀": "Stationary",
        "Conclusion": "Stationary" if kpss_pval > 0.05 else "Non-stationary",
    })

    df_results = pd.DataFrame(results)
    print(f"\n{'=' * 70}")
    print(f"  Stationarity tests — {name}")
    print(f"{'=' * 70}")
    print(df_results.to_string(index=False))

    # Joint interpretation
    adf_stationary = adf_pval < 0.05
    kpss_stationary = kpss_pval > 0.05
    if adf_stationary and kpss_stationary:
        print("\n  → Both tests agree: the series is STATIONARY.")
    elif not adf_stationary and not kpss_stationary:
        print("\n  → Both tests agree: the series is NON-STATIONARY.")
    elif adf_stationary and not kpss_stationary:
        print("\n  → Conflict: ADF indicates stationary, KPSS indicates non-stationary.")
        print("    Probably trend-stationary (stationary around a deterministic trend).")
    else:
        print("\n  → Conflict: ADF indicates non-stationary, KPSS indicates stationary.")
        print("    Probably difference-stationary.")
    print(f"{'=' * 70}")

    return df_results


# Test on the raw demand series (no imputation applied)
_ = stationarity_tests(demand_series, name="Demand")

### 5.2 MSTL decomposition

MSTL with periods 24 and 168 h on the full series; the seasonal panel shows the sum of the daily and weekly seasonal components. A second decomposition on January 2022 only is plotted for readability, so that the daily and weekly patterns can be seen in detail. The residual diagnostics of Section 5.3 use the full-series decomposition.

In [ ]:
# ── 5.2  MSTL decomposition (periods = [24, 168]), full series ───────────────

mstl = MSTL(demand_series.dropna(), periods=[24, 168], stl_kwargs={"robust": True})
mstl_result = mstl.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

components = [
    ("Observed", mstl_result.observed, "steelblue"),
    ("Trend", mstl_result.trend, "darkorange"),
    ("Seasonal (combined)", mstl_result.seasonal.sum(axis=1) if mstl_result.seasonal.ndim > 1 else mstl_result.seasonal, "seagreen"),
    ("Residual", mstl_result.resid, "indianred"),
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.4, color=color)
    ax.set_ylabel("kWh")
    ax.set_title(f"MSTL — {title}", fontsize=11)

axes[-1].set_xlabel("Date")
fig.suptitle("Multiple Seasonal-Trend Decomposition (P=24, P=168)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(Path(RESULTS_DIR, "02_mstl_decomposition.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 5.2  MSTL decomposition (periods = [24, 168]), January 2022 only ─────────
# Separate variable names so that `mstl_result` (full series) is kept for Section 5.3.
demand_2022_jan = demand_series[(demand_series.index.year == 2022) & (demand_series.index.month == 1)].copy()

mstl_jan = MSTL(demand_2022_jan.dropna(), periods=[24, 168], stl_kwargs={"robust": True})
mstl_result_jan = mstl_jan.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

components = [
    ("Observed", mstl_result_jan.observed, "steelblue"),
    ("Trend", mstl_result_jan.trend, "darkorange"),
    ("Seasonal (combined)", mstl_result_jan.seasonal.sum(axis=1) if mstl_result_jan.seasonal.ndim > 1 else mstl_result_jan.seasonal, "seagreen"),
    ("Residual", mstl_result_jan.resid, "indianred"),
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.4, color=color)
    ax.set_ylabel("kWh")
    ax.set_title(f"MSTL — {title}", fontsize=11)

axes[-1].set_xlabel("Date")
fig.suptitle("Multiple Seasonal-Trend Decomposition (P=24, P=168), January 2022", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(Path(RESULTS_DIR, "02_mstl_decomposition_jan2022.png"), dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Stationarity and distribution of the MSTL residual

ADF/KPSS tests on the residual of the full-series MSTL decomposition, together with a histogram against a fitted normal density and a normal Q-Q plot.

In [ ]:
# ── 5.3  Stationarity test on the MSTL residual ──────────────────────────────

mstl_resid = mstl_result.resid.dropna()
_ = stationarity_tests(mstl_resid, name="MSTL Residual")

# Visual check: distribution of the residual
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(mstl_resid.values, bins=100, density=True, alpha=0.7, color="indianred", edgecolor="white")
xmin, xmax = ax.get_xlim()
x = np.linspace(xmin, xmax, 200)
ax.plot(x, sp_stats.norm.pdf(x, mstl_resid.mean(), mstl_resid.std()),
        "k--", linewidth=1.5, label="Normal fit")
ax.set_title("MSTL Residual Distribution")
ax.set_xlabel("Residual (kWh)")
ax.legend()

ax = axes[1]
sp_stats.probplot(mstl_resid.values, dist="norm", plot=ax)
ax.set_title("Q-Q Plot — MSTL Residual vs Normal")

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR, "02_mstl_residual_diagnostics.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Saving the artifacts

Two artifacts are saved:

1. **`data/processed/features.parquet`** — the complete feature DataFrame (including `demand` as the target column) with its hourly `datetime` index. Rows containing NaN due to the initial warm-up of lags/rolling windows *are kept*; each downstream notebook handles them when building its training data.

2. **`data/processed/feature_names.json`** — a dictionary mapping each feature group to its column names, documenting the feature set. (The later notebooks read `features.parquet` directly and do not load this file.)

In [ ]:
# ── 6.1  Save features.parquet ───────────────────────────────────────────────

out_path = Path(DATA_DIR, "processed", "features.parquet")
feat.to_parquet(out_path, engine="pyarrow")
print(f"Saved: {out_path}")
print(f"  Shape     : {feat.shape}")
print(f"  Columns   : {feat.shape[1]}")
print(f"  Size      : {out_path.stat().st_size / 1024**2:.1f} MB")

In [ ]:
# ── 6.2  Save feature_names.json ─────────────────────────────────────────────

json_path = Path(DATA_DIR, "processed", "feature_names.json")
with open(json_path, "w") as f:
    json.dump(feature_names, f, indent=2)

print(f"Saved: {json_path}")
print(json.dumps(feature_names, indent=2))

In [ ]:
# ── 6.3  Final summary ───────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  PREPROCESSING COMPLETED")
print("=" * 70)
print(f"  Original data (hours)   : {len(demand_series):,}")
print(f"  Generated features      : {total_features}")
print(f"  Output rows             : {len(feat):,}")
print(f"  Output columns          : {feat.shape[1]}")
print(f"\n  Files:")
print(f"    → {out_path}")
print(f"    → {json_path}")
print("=" * 70)